## Master Analytics Table & Executive Insights
#### Objective
The goal is to consolidate all analytical outputs into a single, business-ready analytics layer that supports:
* Executive reporting
* Dashboard development
* Cross-segment comparison
* Risk monitoring and decision-making

This phase integrates **financial health scoring, account-level segmentation, and segment profiling** into a unified dataset.

In [1]:
import pandas as pd
import numpy as np

# Financial health scores from Notebook 3
account_financial_health = pd.read_csv("../outputs/account_financial_health.csv")

# Account-level segmentation output from Notebook 4
account_segmentation = pd.read_csv("../outputs/account_segmentation.csv")

# Segment-level profiles
segment_profile = pd.read_csv("../outputs/segment_profile.csv")

In [3]:
# Inspecting schemas
account_financial_health.head()

,Account No,year_month,txn_count,total_inflow,total_outflow,avg_txn_amount,balance_std,unique_merchants,net_cash_flow,avg_balance,low_balance_flag,balance_score,cashflow_score,volatility_score,txn_score,low_balance_score,financial_health_score,health_category
0,100100738,2025-01,17,0.0,-1192.90,-70.170588,495.643012,7,-1192.90,-1192.90,1,0.980307,0.295761,0.910529,0.25000,0.0,58.763785,Moderate
1,100100738,2025-02,9,0.0,-834.03,-92.670000,313.924862,7,-834.03,-2026.93,1,0.979875,0.301960,0.943332,0.12500,0.0,57.686884,Moderate
2,100100738,2025-03,3,0.0,-155.98,-51.993333,132.771179,2,-155.98,-2182.91,1,0.979794,0.313674,0.976033,0.03125,0.0,57.225072,Moderate
3,100100738,2025-04,3,0.0,-241.83,-80.610000,557.072221,2,-241.83,-2424.74,1,0.979669,0.312191,0.899440,0.03125,0.0,55.652378,Moderate
4,100100738,2025-05,1,0.0,-110.52,-110.520000,NaN,1,-110.52,-2535.26,1,0.979612,0.314459,1.000000,0.00000,0.0,57.249830,Moderate


In [5]:
account_segmentation.head()

,Account No,avg_txn_count,avg_cashflow,avg_balance,avg_volatility,avg_unique_merchants,low_balance_rate,segment
0,100100738,4.250000,-318.331667,-2787.235833,468.764740,2.583333,1.0,1
1,100837224,2.363636,-235.625455,-5342.044545,200.300186,1.181818,1.0,1
2,101348775,15.333333,1074.268333,9560.719167,1147.342711,8.416667,0.0,2
3,103439190,19.166667,1867.390000,32648.592500,2493.829571,11.000000,0.0,2
4,104009728,13.833333,-1436.545833,14603.865000,1909.106936,9.750000,0.0,2


In [7]:
segment_profile.head()

,Unnamed: 0,segment,avg_txn_count,avg_cashflow,avg_balance,avg_volatility,avg_unique_merchants,low_balance_rate
0,0,0,13.37,398.96,-1017481.80,1542.88,7.77,1.0
1,1,1,6.08,-338.23,-985556.84,658.39,3.23,1.0


In [9]:
account_financial_health["Account No"].nunique()

1210

In [11]:
account_segmentation["Account No"].nunique()

1210

In [13]:
# Creating the Master Analytics Table for Dashboard & Insights
master_df = account_financial_health.merge(
    account_segmentation[[
            "Account No",
            "segment",
            "avg_txn_count",
            "avg_cashflow",
            "avg_balance",
            "avg_volatility",
            "avg_unique_merchants",
            "low_balance_rate"
        ]],
    on="Account No",
    how="left"
)

master_df.columns

Index(['Account No', 'year_month', 'txn_count', 'total_inflow',
       'total_outflow', 'avg_txn_amount', 'balance_std', 'unique_merchants',
       'net_cash_flow', 'avg_balance_x', 'low_balance_flag', 'balance_score',
       'cashflow_score', 'volatility_score', 'txn_score', 'low_balance_score',
       'financial_health_score', 'health_category', 'segment', 'avg_txn_count',
       'avg_cashflow', 'avg_balance_y', 'avg_volatility',
       'avg_unique_merchants', 'low_balance_rate'],
      dtype='object')

In [15]:
master_df = master_df.rename(columns={
    "avg_balance_x": "monthly_avg_balance",
    "avg_balance_y": "account_avg_balance"
})

In [17]:
master_df["segment"].isna().mean()

0.0

In [19]:
# Renaming the Numeric Segment IDs to Business Friendly Segment Names
segment_map = {
    0: "Financially Strained",
    1: "Stable Low-Engagement"
}

master_df["segment_name"] = master_df["segment"].map(segment_map)

In [21]:
master_df = master_df[
    [
        "Account No",
        "year_month",
        "segment",
        "segment_name",
        "financial_health_score",
        "health_category",
        "monthly_avg_balance",
        "account_avg_balance",
        "net_cash_flow",
        "txn_count",
        "avg_txn_amount",
        "balance_std",
        "avg_txn_count",
        "avg_cashflow",
        "avg_volatility",
        "avg_unique_merchants",
        "low_balance_rate",
        "low_balance_flag"
    ]
]
master_df.columns

Index(['Account No', 'year_month', 'segment', 'segment_name',
       'financial_health_score', 'health_category', 'monthly_avg_balance',
       'account_avg_balance', 'net_cash_flow', 'txn_count', 'avg_txn_amount',
       'balance_std', 'avg_txn_count', 'avg_cashflow', 'avg_volatility',
       'avg_unique_merchants', 'low_balance_rate', 'low_balance_flag'],
      dtype='object')

In [23]:
# Executive Summary Metrics
# What should leadership know in 30 seconds?

executive_summary = (
    master_df
    .groupby("segment_name")
    .agg(
        accounts=("Account No", "nunique"),
        avg_health_score=("financial_health_score", "mean"),
        at_risk_rate=("health_category", lambda x: (x == "At-Risk").mean()),
        avg_account_balance=("account_avg_balance", "mean"),
        avg_account_cashflow=("avg_cashflow", "mean")
    )
    .reset_index()
)

executive_summary

,segment_name,accounts,avg_health_score,at_risk_rate,avg_account_balance,avg_account_cashflow
0,Financially Strained,285,39.065639,0.595196,-1.017122e+06,397.532074
1,Stable Low-Engagement,921,41.128482,0.524324,-9.846124e+05,-322.544285


### Executive Summary Metrics
To support rapid executive decision-making, segment-level summary metrics are computed.

#### Key Questions Answered
1. How many accounts belong to each segment?
2. Which segments carry higher financial risk?
3. How do financial health and balances differ by segment?
4. Where should attention or intervention be prioritized?

#### Executive Metrics Computed
1. Total number of accounts per segment
2. Average financial health score
3. Percentage of accounts classified as At-Risk
4. Average account balance
5. Average account-level cashflow

These metrics are aggregated at the **segment level** and saved as a standalone artifact for reporting.

In [25]:
executive_summary["at_risk_rate"] = (
    executive_summary["at_risk_rate"] * 100
).round(1)

In [27]:
master_df.to_csv(
    "../outputs/master_account_analytics.csv",
    index=False
)

executive_summary.to_csv(
    "../outputs/executive_summary.csv",
    index=False
)